# Phase 11 — tau-bench-inspired subset, baseline
**Scope note:** this is NOT the official tau-bench harness (which needs its own package, a user-simulator LLM, and database-state-diff scoring). This hand-ports real tau-bench airline/retail tool signatures into this project's existing deterministic-simulator + grader style, as an honest, modest external-domain generalization check.

Baseline only (no DSPy/QLoRA optimization) on Llama-3.1-8B-Instruct, matching Phase 10's minimum-scope precedent. Single session, no restart needed.

**Before running:** Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!nvidia-smi

**STOP: must show 0MiB used before continuing.**

## 1. Clone repo

In [ ]:
import shutil, os
os.chdir("/content")
if os.path.exists("agentic-prompt-vs-finetune"):
    shutil.rmtree("agentic-prompt-vs-finetune")
!git clone https://github.com/nive62tech/agentic-prompt-vs-finetune.git
%cd agentic-prompt-vs-finetune
!pip install -q transformers accelerate bitsandbytes

In [ ]:
from huggingface_hub import login
login()

## 2. Sanity-check the grader locally first (no GPU needed)

In [ ]:
!python grader_taubench.py

All 14 lines above should say PASS. If any FAIL, stop and check before using GPU time.

## 3. Load model

In [ ]:
import sys, json
sys.path.insert(0, ".")
from envs.agent_harness import load_model, run_agent
from envs.tools_taubench import TOOL_SCHEMAS, call_tool
from grader_taubench import grade_taubench_task

model, tok = load_model("meta-llama/Llama-3.1-8B-Instruct")
print("Model loaded.")

## 4. Chain tasks (multi-tool, unordered pairs) — full set + held-out

In [ ]:
from tasks.taubench_subset import TAUBENCH_CHAIN_TASKS, TAUBENCH_CHAIN_HELDOUT

all_chain = TAUBENCH_CHAIN_TASKS + TAUBENCH_CHAIN_HELDOUT
chain_results = []
for task in all_chain:
    tool_calls, final_text = run_agent(model, tok, task["prompt"], TOOL_SCHEMAS, call_tool, max_turns=6)
    grade = grade_taubench_task(task, tool_calls, final_text)
    chain_results.append({"id": task["id"], "prompt": task["prompt"], "tool_calls": tool_calls, "final_text": final_text, "grade": grade})
    print(f"[{'PASS' if grade['success'] else 'FAIL'}] {task['id']} — {grade.get('failure_type')}")

rate = sum(r["grade"]["success"] for r in chain_results) / len(chain_results)
print(f"\ntau-bench-subset chain tasks baseline: {rate:.1%}")

with open("results/taubench_chain_baseline_results.json", "w") as f:
    json.dump(chain_results, f, indent=2)

## 5. Error-recovery tasks — full set + held-out

In [ ]:
from tasks.taubench_subset import TAUBENCH_ERROR_TASKS, TAUBENCH_ERROR_HELDOUT

all_error = TAUBENCH_ERROR_TASKS + TAUBENCH_ERROR_HELDOUT
error_results = []
for task in all_error:
    tool_calls, final_text = run_agent(model, tok, task["prompt"], TOOL_SCHEMAS, call_tool, max_turns=6)
    grade = grade_taubench_task(task, tool_calls, final_text)
    error_results.append({"id": task["id"], "prompt": task["prompt"], "tool_calls": tool_calls, "final_text": final_text, "grade": grade})
    print(f"[{'PASS' if grade['success'] else 'FAIL'}] {task['id']} — {grade.get('failure_type')}")

rate = sum(r["grade"]["success"] for r in error_results) / len(error_results)
print(f"\ntau-bench-subset error-recovery tasks baseline: {rate:.1%}")

with open("results/taubench_error_baseline_results.json", "w") as f:
    json.dump(error_results, f, indent=2)

## 6. Download and push

In [ ]:
from google.colab import files
files.download("results/taubench_chain_baseline_results.json")
files.download("results/taubench_error_baseline_results.json")

Move both into `results/` on your laptop, then:
```bash
git add results/taubench_chain_baseline_results.json results/taubench_error_baseline_results.json
git commit -m "Phase 11: tau-bench-inspired subset baseline results"
git push
```

## What this tells us
Compare these rates to Tier 2 (chain-style, 41.7% baseline on the original suite) and Tier 3 (error-recovery, 90.0% baseline). If the pattern is similar, that's evidence the original findings aren't an artifact of this project's specific simulated weather/currency/flight domain. If very different, that itself is informative — same lesson as Phase 10's Qwen check: generalization needs to be checked, not assumed.